# Composite Original vs Unbiased Bias Summary Plots

Each condition has one bar position per emotion. For values with the same sign, the shared magnitude is saturated and the difference to the higher-magnitude value is lighter. For values with opposite signs, the original value is saturated and the unbiased value is lighter. Endpoint tags identify the datasets: `O` for original and `UB` for unbiased.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
import seaborn as sns

try:
    import openpyxl  # noqa: F401
except ImportError as exc:
    raise ImportError(
        'openpyxl is required to read Excel files. Install it with `pip install openpyxl`.'
    ) from exc

sns.set(style='whitegrid', font_scale=1.1)
%matplotlib inline

In [ ]:
dataset_files = {
    'Original': Path('bias_summary_10folds_biased.xlsx'),
    'Unbiased': Path('bias_summary_10folds_unbiased.xlsx'),
}

dataset_frames = {}
for dataset, file_path in dataset_files.items():
    if not file_path.exists():
        raise FileNotFoundError(f'Expected file not found: {file_path.resolve()}')

    try:
        workbook = pd.ExcelFile(file_path, engine='openpyxl')
        frame = pd.read_excel(workbook, sheet_name=workbook.sheet_names[0], engine='openpyxl')
    except Exception as exc:
        raise RuntimeError(
            f'Unable to read {file_path}. Close it in Excel and rerun the cell.'
            f' Original error: {exc}'
        ) from exc

    dataset_frames[dataset] = frame
    print(f'{dataset}: {file_path.name} ({len(frame)} conditions)')

In [ ]:
emotion_keys = ['happy', 'angry', 'sad', 'neutral']
emotion_labels = [emotion.title() for emotion in emotion_keys]
required_cols = [
    'condition',
    *[f'{bias}_{emotion}' for bias in ['sp', 'eo'] for emotion in emotion_keys],
]

for dataset, frame in dataset_frames.items():
    missing_cols = [col for col in required_cols if col not in frame.columns]
    if missing_cols:
        raise KeyError(f'Missing columns in {dataset} dataset: {missing_cols}')

condition_order = dataset_frames['Original']['condition'].drop_duplicates().tolist()
condition_labels = {
    'azz': 'A',
    'zvz': 'V',
    'zzl': 'T',
    'avz': 'A+V',
    'azl': 'A+T',
    'zvl': 'V+T',
    'avl': 'A+V+T',
}
for dataset, frame in dataset_frames.items():
    conditions = frame['condition'].drop_duplicates().tolist()
    if conditions != condition_order:
        raise ValueError(
            f'Condition order differs in {dataset} dataset: {conditions}. '
            f'Expected: {condition_order}'
        )

values = {}
for bias in ['SP', 'EO']:
    values[bias] = {}
    for emotion in emotion_keys:
        column = f'{bias.lower()}_{emotion}'
        values[bias][emotion] = {}
        for condition in condition_order:
            values[bias][emotion][condition] = {
                dataset: float(
                    dataset_frames[dataset].loc[
                        dataset_frames[dataset]['condition'] == condition, column
                    ].iloc[0]
                )
                for dataset in dataset_files
            }

print('Conditions:', condition_order)

In [ ]:
def darken(color, amount=0.72):
    return tuple(channel * amount for channel in color)


def lighten(color, amount=0.32):
    return tuple(channel + (1 - channel) * amount for channel in color)


base_colors = dict(
    zip(condition_order, sns.color_palette('Set3', n_colors=len(condition_order)))
)
saturated_colors = {
    condition: darken(color) for condition, color in base_colors.items()
}
lighter_colors = {
    condition: lighten(color) for condition, color in base_colors.items()
}


def annotate_endpoint(ax, x, value, tag, x_offset):
    y_offset = 3 if value >= 0 else -3
    vertical_alignment = 'bottom' if value >= 0 else 'top'
    ax.annotate(
        tag,
        (x, value),
        xytext=(x_offset, y_offset),
        textcoords='offset points',
        ha='center',
        va=vertical_alignment,
        fontsize=7,
        fontweight='bold',
    )


def draw_composite_bar(ax, x, original, unbiased, saturated, lighter, width):
    same_direction = original * unbiased >= 0
    if same_direction:
        lower_magnitude = min(abs(original), abs(unbiased))
        higher_magnitude = max(abs(original), abs(unbiased))
        direction = -1 if original < 0 or unbiased < 0 else 1
        shared_height = direction * lower_magnitude
        difference_height = direction * (higher_magnitude - lower_magnitude)

        ax.bar(x, shared_height, width=width, color=saturated)
        if difference_height != 0:
            ax.bar(x, difference_height, width=width, bottom=shared_height, color=lighter)
    else:
        ax.bar(x, original, width=width, color=saturated)
        ax.bar(x, unbiased, width=width, color=lighter)

    annotate_endpoint(ax, x, original, 'O', -4)
    annotate_endpoint(ax, x, unbiased, 'UB', 5)


def plot_bias(bias, title):
    fig, ax = plt.subplots(figsize=(15, 6))
    condition_count = len(condition_order)
    group_width = 0.82
    bar_width = group_width / condition_count

    for emotion_index, emotion in enumerate(emotion_keys):
        start_x = emotion_index - group_width / 2 + bar_width / 2
        for condition_index, condition in enumerate(condition_order):
            x = start_x + condition_index * bar_width
            pair = values[bias][emotion][condition]
            draw_composite_bar(
                ax=ax,
                x=x,
                original=pair['Original'],
                unbiased=pair['Unbiased'],
                saturated=saturated_colors[condition],
                lighter=lighter_colors[condition],
                width=bar_width * 0.92,
            )

    condition_legend = [
        Patch(facecolor=saturated_colors[condition], label=condition_labels[condition])
        for condition in condition_order
    ]
    shade_legend = [
        Patch(facecolor='gray', label='Saturated: shared value / original if signs differ'),
        Patch(facecolor='lightgray', label='Lighter: difference / unbiased if signs differ'),
    ]

    ax.set_title(title)
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Difference')
    ax.set_xticks(range(len(emotion_labels)), emotion_labels)
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
    ax.legend(
        handles=condition_legend + shade_legend,
        title='Condition and Shade Meaning',
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
    )
    plt.tight_layout()
    return ax

In [ ]:
sp_ax = plot_bias(
    bias='SP',
    title='Statistical Parity Difference by Emotion: Original vs Unbiased Dataset',
)
plt.show()

In [ ]:
eo_ax = plot_bias(
    bias='EO',
    title='Equal Opportunity Difference by Emotion: Original vs Unbiased Dataset',
)
plt.show()

## Save the Plots

In [ ]:
# sp_ax.figure.savefig('bias_summary_sp_composite_original_vs_unbiased.png', dpi=150, bbox_inches='tight')
# eo_ax.figure.savefig('bias_summary_eo_composite_original_vs_unbiased.png', dpi=150, bbox_inches='tight')